In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import load_workbook

In [2]:
folder = Path("../Sqllab Missing Value NIK Terbaru/")
files = list(folder.glob("sqllab_sumsel_missing_value_nik*.xlsx"))

if not files:
    raise FileNotFoundError("Tidak ada file Excel di folder scrap_anomali_usaha")


print(f"Ditemukan {len(files)} file:")
for f in files:
    print(" -", f.name)

# Baca semua file lalu gabungkan
list_df = []
for f in files:
    df = pd.read_excel(f, dtype=str)
    df["source_file"] = (
        f.name
    )  # opsional: kolom penanda asal file, bisa dihapus kalau tidak perlu
    list_df.append(df)

df_gabungan = pd.concat(list_df, ignore_index=True)

print(f"\nTotal baris setelah digabung: {len(df_gabungan)}")
df_gabungan.head()

Ditemukan 12 file:
 - sqllab_sumsel_missing_value_nik.xlsx
 - sqllab_sumsel_missing_value_nik10.xlsx
 - sqllab_sumsel_missing_value_nik11.xlsx
 - sqllab_sumsel_missing_value_nik12.xlsx
 - sqllab_sumsel_missing_value_nik2.xlsx
 - sqllab_sumsel_missing_value_nik3.xlsx
 - sqllab_sumsel_missing_value_nik4.xlsx
 - sqllab_sumsel_missing_value_nik5.xlsx
 - sqllab_sumsel_missing_value_nik6.xlsx
 - sqllab_sumsel_missing_value_nik7.xlsx
 - sqllab_sumsel_missing_value_nik8.xlsx
 - sqllab_sumsel_missing_value_nik9.xlsx

Total baris setelah digabung: 11223


,rn,assignment_id,level_6_full_code,no_keluarga,no_bang,no_kk,no_urut_kk,nama_kk,nama_ak_lain,nama_dtsen,nik_dtsen,source_file
0,1,7c37a227-0f01-4dda-8797-346424f2b487,1601052001000100,9,9,1601281907190001,6,DARMA ANITA,REZI,REFI NATASYA WILONA,9999,sqllab_sumsel_missing_value_nik.xlsx
1,2,e73100b0-2479-420d-8f67-963d2c97e0e0,1601052001000100,12,10,1601281310100001,4,MERTA RIA,IRA YANA,AISYAH AZZAHRA,9999,sqllab_sumsel_missing_value_nik.xlsx
2,3,fde573ae-f303-46a8-8db5-15cee2777032,1601052001000100,40,33,1601283011100011,5,SAPTA DARYANDI,AYU AGUSTIN,SHANUM YUMNA ARADEA,9999,sqllab_sumsel_missing_value_nik.xlsx
3,4,854b21bc-6a5f-4a7e-9efe-9e25aafe40c3,1601052001000100,45,38,1601281103080001,1,SUMAIDAH,SUMAIDAH,ERMIN NIZA,9999,sqllab_sumsel_missing_value_nik.xlsx
4,5,5462ce1c-1c11-4215-9888-473b4f20030a,1601052001000700,37,27,1601280601090004,4,SAPARUDIN,MINARNI,Joni,9999,sqllab_sumsel_missing_value_nik.xlsx


In [3]:
# Kolom yang jadi patokan unik
kolom_unik = [
    "assignment_id",
    "level_6_full_code",
    "no_keluarga",
    "no_bang",
    "no_kk",
    "no_urut_kk",
    "nama_kk",
    "nama_dtsen",
]

# Hapus duplikat, simpan kemunculan pertama
df_final = (
    df_gabungan.drop_duplicates(subset=kolom_unik, keep="first")
    .reset_index(drop=True)
    .rename(columns={"rank_keluarga": "nomor"})
    .drop(columns=["source_file"])
)

print(f"Total baris setelah dedup: {len(df_final)}")
print(f"Jumlah baris duplikat yang dihapus: {len(df_gabungan) - len(df_final)}")

df_final.head()

Total baris setelah dedup: 11221
Jumlah baris duplikat yang dihapus: 2


,rn,assignment_id,level_6_full_code,no_keluarga,no_bang,no_kk,no_urut_kk,nama_kk,nama_ak_lain,nama_dtsen,nik_dtsen
0,1,7c37a227-0f01-4dda-8797-346424f2b487,1601052001000100,9,9,1601281907190001,6,DARMA ANITA,REZI,REFI NATASYA WILONA,9999
1,2,e73100b0-2479-420d-8f67-963d2c97e0e0,1601052001000100,12,10,1601281310100001,4,MERTA RIA,IRA YANA,AISYAH AZZAHRA,9999
2,3,fde573ae-f303-46a8-8db5-15cee2777032,1601052001000100,40,33,1601283011100011,5,SAPTA DARYANDI,AYU AGUSTIN,SHANUM YUMNA ARADEA,9999
3,4,854b21bc-6a5f-4a7e-9efe-9e25aafe40c3,1601052001000100,45,38,1601281103080001,1,SUMAIDAH,SUMAIDAH,ERMIN NIZA,9999
4,5,5462ce1c-1c11-4215-9888-473b4f20030a,1601052001000700,37,27,1601280601090004,4,SAPARUDIN,MINARNI,Joni,9999


In [4]:
output_folder = Path("../Rekap Missing Value")
output_folder.mkdir(parents=True, exist_ok=True)

df_final["kode_kabupaten"] = df_final["level_6_full_code"].astype(str).str[:4]
df_ringkasan = (
    df_final.groupby("kode_kabupaten")
    .size()
    .reset_index(name="jumlah_row")
    .sort_values("kode_kabupaten")
)

# Hapus kolom bantu kode_kabupaten dari sheet utama kalau tidak ingin ikut tampil
# (hapus baris ini kalau justru ingin kolom ini tetap ada di sheet 1)
df_final = df_final.drop(columns=["kode_kabupaten"])

# Nama file dengan timestamp sekarang
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"sumsel_missing_nik_{timestamp}.xlsx"
filepath = output_folder / filename
with pd.ExcelWriter(filepath, engine="openpyxl") as writer:
    df_final.to_excel(writer, sheet_name="Data", index=False)
    df_ringkasan.to_excel(writer, sheet_name="Ringkasan", index=False)

# Auto-adjust lebar kolom
wb = load_workbook(filepath)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for col_cells in ws.columns:
        max_length = max(
            len(str(cell.value)) if cell.value is not None else 0 for cell in col_cells
        )
        col_letter = col_cells[0].column_letter
        ws.column_dimensions[col_letter].width = min(
            max_length + 3, 50
        )  # +3 padding, max 50 biar tidak kebablasan
wb.save(filepath)

print(f"File tersimpan di: {filepath.resolve()}")

File tersimpan di: D:\2026\Sensus Ekonomi\Rekap Missing Value\sumsel_missing_nik_20260710_091120.xlsx
